In [11]:
import pandas as pd
import numpy as np
import sys, os
from helpers import Condition, Participant, Trial
from helpers.metadata import AVOIDING_THRESHOLD,PIXEL_RAILWAY_WIDTH

In [12]:
ai = Condition.load_conditional_group(condition="ai")
ad = Condition.load_conditional_group(condition="ad")

In [13]:
def transform_to_normalized_distances(trial: Trial) -> Trial:
    """Nomalize to start = 0 and target = 1"""

    tx, ty = trial.get_target_in_cm()
    sx, _ = trial.get_start()
    trajectory_px = trial.get_trajectory_data()[["current_pos_x"]]

    trial[["current_pos_x_cm", "current_pos_y_cm"]] = trial.get_trajectory_data_in_cm()[["current_pos_x", "current_pos_y"]]
    trial[["current_pos_x_normalized", "current_pos_y_normalized"]] = trial.get_trajectory_data_normalized()[["current_pos_x", "current_pos_y"]]
    
    trial[["target_pos_x_cm"]] = tx
    trial[["target_pos_y_cm"]] = ty
    trial["trial_rail_pos"] = (trajectory_px + np.abs(sx)) / PIXEL_RAILWAY_WIDTH

    return trial

In [14]:
threshold = AVOIDING_THRESHOLD * PIXEL_RAILWAY_WIDTH
def add_trajectory_side(trial:
     Trial) -> Trial:
    """appends trial dataframe by column to store whether side was switched"""

    trial["switched_side"] = 0 
    trial.loc[trial["trial_rail_pos"] >= AVOIDING_THRESHOLD, "switched_side"] = 1
    return trial

In [15]:
def drop_unneccessary(trial: Trial):
    trial = trial.drop(["left_button_pressed","middle_button_pressed","right_button_pressed", "current_pos_x", "current_pos_y", "target_pos_x", "target_pos_y"], axis=1)
    return trial

In [16]:
#def normalize_time(data):
#    """
#    Normalize the time frame to go from 0 to 1.
#    Interpolate the data to get 10ms time steps
#    """

#    data.loc[:,'datetime'] = pd.date_range('1/1/2001 00:00:00', '1/1/2001 00:00:01',len(data))
#    normdata = data.set_index('datetime', drop = True).resample('10ms').mean().interpolate()
#    normdata['normtime'] = np.arange(0,1.01,0.01)
#    normdata = normdata.reset_index(drop=True)

#    return normdata

In [17]:
def get_relevant_row(trial: Trial) -> pd.DataFrame:
    """Gets the actual row per trial"""

    trial = transform_to_normalized_distances(trial)
    trial = add_trajectory_side(trial)
    type = trial.loc[:, "task"].iloc[0]
    
    if type == "avoiding":
        switched = trial.loc[trial["switched_side"] == 1]
        return switched.iloc[0] if len(switched) > 0 else trial.iloc[-1]
    elif type == "reaching":
        return trial.iloc[-1]
    else:
        raise ValueError(f"Faulty trial type found! No typ of {type}")


def find_natural_mapping(trials: list[Trial]):
    """Finding natural mapping by identifiying correlation between target and actual. negative means higher target leads to lower estimate"""
    
    relevant_rows = list(map(get_relevant_row, trials))
    pos = np.array(list(map(lambda trial: np.array([trial[["target_pos_y"]].iloc[-1], trial[["current_pos_y"]].iloc[-1]]),  relevant_rows)))
    
    X = pos[:, 0]
    y = pos[:, 1]
    corr = np.corrcoef(X, y)[0, 1] # returns covariance matrix
    
    task_mapping = trials[0].get_mapping()

    THRESHOLD = .33
    if (task_mapping == "reversed" and corr >= THRESHOLD) or (task_mapping == "direct" and corr <= THRESHOLD):
        return "reversed"
    elif (task_mapping == "direct" and corr >= THRESHOLD) or (task_mapping == "reversed" and corr <= THRESHOLD):
        return "direct"
    else:
        return "random"
    

In [18]:
def process(trial:Trial, natural_mapping: str) -> Trial:
    trial = transform_to_normalized_distances(trial)
    trial = add_trajectory_side(trial)
    trial = drop_unneccessary(trial)
    trial["natural_mapping"] = natural_mapping
    
    return trial


In [19]:
FILE_PATH = "preprocessed_avoiding"
if not os.path.exists(FILE_PATH):
    os.mkdir(FILE_PATH)

for condition in [ai, ad]:

    p_processed = []
    for participant in condition:
        sys.stdout.write(f"\rProcessing {participant.get_participant_id()}\t")

        pre_test = participant.get_as_one_list(phase="Pre-Test", block=1)
        natural_mapping = find_natural_mapping(pre_test)

        p_processed += list(map(lambda trial: process(trial, natural_mapping), participant))

    sys.stdout.write(f"\rConcatinating {condition.get_group_name()}\t")
    df_joint = pd.concat(p_processed).round(3)
    df_joint.to_csv(f"{FILE_PATH}/{condition.get_group_name()}.csv")


sys.stdout.write(f"\nDone")
sys.stdout.flush()

Concatinating ad	
Done